# Use `Tabular` for Customs Datasets

# Installing Anomalib

The easiest way to install anomalib is to use pip. You can install it from the command line using the following command:


In [ ]:
%pip install anomalib

In [ ]:
import io
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from torchvision.transforms.v2.functional import to_pil_image

from anomalib.data import Tabular, TabularDataset

## Setting up the Dataset Directory

Prepare dataset folder and configure `dataset_root`:


In [ ]:
# Download the hazelnut toy dataset if it doesn't exist in the specified directory
dataset_root = Path.cwd().parent.parent.parent.parent / "datasets" / "hazelnut_toy"
if dataset_root.exists():
    print(f"Dataset already initialized at {dataset_root}")
else:
    dataset_root.parent.mkdir(parents=True, exist_ok=True)
    with (
        urllib.request.urlopen(
            "https://github.com/open-edge-platform/anomalib/releases/download/hazelnut_toy_dataset/hazelnut_toy.zip",
        ) as response,
        zipfile.ZipFile(io.BytesIO(response.read())) as z,
    ):
        z.extractall(dataset_root.parent)

## Use Tabular Dataset (for Custom Datasets) via API

### DataModule

Similar to the Folder datamodule, we can use the Tabular datamodule to create a custom hazelnut toy dataset.

Furthermore, the Tabular datamodule allows defining custom subsets of the files present in the folder structure, and it is possible to manage image labels independent from the folder structure (useful for label refinement).

In [ ]:
# Create samples dataframe from folder structure. In practice, this information could be stored in a database
# or tabular file and does not need to be encoded in the folder structure.
normal_image_paths = sorted([
    str(file_path.relative_to(dataset_root)) for file_path in Path(dataset_root / "good").iterdir()
])
anomalous_image_paths = sorted([
    str(file_path.relative_to(dataset_root)) for file_path in Path(dataset_root / "crack").iterdir()
])
mask_paths = [str(Path("mask") / file_path) for file_path in anomalous_image_paths]
samples = pd.DataFrame({
    "image_path": normal_image_paths + anomalous_image_paths,
    "mask_path": [None] * len(normal_image_paths) + mask_paths,
    "label_index": [0] * len(normal_image_paths) + [1] * len(anomalous_image_paths),
    "split": ["train"] * len(normal_image_paths) + ["test"] * len(anomalous_image_paths),
})

# Create a Tabular datamodule
tabular_datamodule = Tabular(
    name="hazelnut_toy",
    samples=samples,
    root=dataset_root,
)
tabular_datamodule.setup()

In [ ]:
# Train images
train_sample = next(iter(tabular_datamodule.train_data))
print(train_sample.image.shape)

In [ ]:
# Test images
test_sample = next(iter(tabular_datamodule.test_data))
print(test_sample.image.shape, test_sample.gt_mask.shape)

As can be seen above, creating the dataloaders are pretty straghtforward, which could be directly used for training/testing/inference. We could visualize samples from the dataloaders as well.


In [ ]:
img = to_pil_image(test_sample.image.clone())
msk = to_pil_image(test_sample.gt_mask.int() * 255).convert("RGB")
Image.fromarray(np.hstack((np.array(img), np.array(msk))))

### Torch Dataset

As in earlier examples, we can also create a standalone PyTorch dataset instance.


In [ ]:
TabularDataset??

Now let's create the dataset, we'll start with the training subset (we drop the `mask_path` column to simulate the classification scenario).

In [ ]:
# Tabular Classification Train Set
classification_train = TabularDataset(
    name="hazelnut_toy",
    samples=samples.drop(columns=["mask_path"]),
    root=dataset_root,
    split="train",
)
print(len(classification_train))
train_sample = classification_train[0]
print(train_sample.image.shape, train_sample.image_path, train_sample.gt_label)

As can be seen above, when we choose `train` split, the dataset contains 34 samples. These are the normal images that have been assigned to the training set, which have a corresponding ground truth label of `False`, indicating that the image does not contain an anomaly. 

Now let's have a look at the test set which also contains samples with a ground truth label of `True`, indicating an anomaly.

In [ ]:
# Tabular Classification Test Set
classification_test = TabularDataset(
    name="hazelnut_toy",
    samples=samples.drop(columns=["mask_path"]),
    root=dataset_root,
    split="test",
)
print(len(classification_test))
test_sample = classification_test[0]
print(test_sample.image.shape, test_sample.image_path, test_sample.gt_label)

#### Segmentation Task

It is also possible to configure the Tabular dataset for the segmentation task, where the dataset object returns image and ground-truth mask. To achieve this, we need include a `mask_path` column in the `samples` dataframe. The `mask_path` column should contain the path to a ground truth pixel mask for every anomalous image in the dataset.


In [ ]:
# Tabular Segmentation Train Set
segmentation_train = TabularDataset(
    name="hazelnut_toy",
    samples=samples,
    root=dataset_root,
    split="train",
)
print(len(segmentation_train))
train_sample = segmentation_train[0]
print(train_sample.image.shape, train_sample.gt_mask.shape, train_sample.image_path, train_sample.gt_label)

In [ ]:
# Tabular Segmentation Test Set
segmentation_test = TabularDataset(
    name="hazelnut_toy",
    samples=samples,
    root=dataset_root,
    split="test",
)
print(len(segmentation_test))
test_sample = segmentation_test[0]
print(test_sample.image.shape, test_sample.gt_mask.shape, test_sample.image_path, test_sample.gt_label)

Let's visualize the image and the mask...


In [ ]:
img = to_pil_image(test_sample.image.clone())
msk = to_pil_image(test_sample.gt_mask.int() * 255).convert("RGB")
Image.fromarray(np.hstack((np.array(img), np.array(msk))))